# Pipeline (temporário) — T-102 a T-107

Notebook de revisão: encadeia as funções de `src/` já implementadas sobre os
dados reais, em ordem, com checagens de sanidade em cada etapa. **Não é o
notebook final de EDA (T-110)** — serve para revisar o pipeline até aqui
antes de seguir para o contrato e a escrita (T-108).

Lógica vive em `src/`; este notebook só chama e exibe.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from src import attribution, clean, config, cost, features, io

spark = (
    SparkSession.builder.appName("ifood-pipeline-review")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/08 23:16:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Config

Carregada de `config.yaml` na raiz do projeto (REQ-110).

In [2]:
cfg = config.load(config_path=Path.cwd().parent / "config.yaml", raw_dir=Path.cwd().parent / "data" / "raw")
cfg

_PipelineConfigWithPath(raw_dir=PosixPath('/home/caioolubini/Projects/ifood-coupons-uplift/data/raw'), processed_dir=PosixPath('data/processed'), offers_filename='offers.json', profile_filename='profile.json', transactions_filename='transactions.json', age_sentinel=118, test_start_date='20180726', smd_threshold=0.1, attribution_priority=<AttributionPriority.EARLIEST_RECEIVED: 'earliest_received'>, campaign_wave_days=5.0, n_campaign_waves=6, seed=42)

## 2. Leitura e parsing de `value` (T-102)

Coalescência `offer id` / `offer_id` em `offer_ref`. Checagem: `received`/`viewed`
não podem perder a referência de oferta; `transaction` não deve ter nenhuma.

In [3]:
events = io.parse_events(spark, cfg)
offers = spark.read.option("multiLine", True).json(str(cfg.offers_path))
profile_raw = spark.read.option("multiLine", True).json(str(cfg.profile_path))

events.groupBy("event").count().orderBy(F.desc("count")).show(truncate=False)

+---------------+------+
|event          |count |
+---------------+------+
|transaction    |138953|
|offer received |76277 |
|offer viewed   |57725 |
|offer completed|33579 |
+---------------+------+



In [4]:
print("linhas totais:", events.count())

# sanidade: offer_ref nulo por tipo de evento
events.groupBy("event").agg(
    F.sum(F.col("offer_ref").isNull().cast("int")).alias("offer_ref_nulo"),
    F.count("*").alias("total"),
).orderBy("event").show(truncate=False)

linhas totais: 306534


+---------------+--------------+------+
|event          |offer_ref_nulo|total |
+---------------+--------------+------+
|offer completed|0             |33579 |
|offer received |0             |76277 |
|offer viewed   |0             |57725 |
|transaction    |138953        |138953|
+---------------+--------------+------+



Esperado: `offer received` e `offer viewed` com `offer_ref_nulo = 0`; `transaction` com `offer_ref_nulo = total`.

In [5]:
offers.show(truncate=False)
offers.groupBy("offer_type").count().show()

+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|channels                    |discount_value|duration|id                              |min_value|offer_type   |
+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|[email, mobile, social]     |10            |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10       |bogo         |
|[web, email, mobile, social]|10            |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10       |bogo         |
|[web, email, mobile]        |0             |4.0     |3f207df678b143eea3cee63160fa8bed|0        |informational|
|[web, email, mobile]        |5             |7.0     |9b98b8c7a33c4b65b9aebfe6a799e6d9|5        |bogo         |
|[web, email]                |5             |10.0    |0b1e1539f2cc45b7b9fa7c272da2e1d7|20       |discount     |
|[web, email, mobile, social]|3             |7.0     |2298d6c36e964ae4a3e7e9706d1fb8c2|7        |discoun

+-------------+-----+
|   offer_type|count|
+-------------+-----+
|         bogo|    4|
|     discount|    4|
|informational|    2|
+-------------+-----+



## 3. Normalização de perfil (T-103)

Sentinela `age=118` → `identity_missing=1`, `age=null`; `gender` ausente → `unknown`;
`tenure_days` a partir de `registered_on`.

In [6]:
profile = clean.normalize_profile(profile_raw, cfg)
profile.show(5, truncate=False)

+--------------------------------+----+-------+-----------------+----------------+-----------+
|account_id                      |age |gender |credit_card_limit|identity_missing|tenure_days|
+--------------------------------+----+-------+-----------------+----------------+-----------+
|68be06ca386d4c31939f3a4f0e3dd783|NULL|unknown|NULL             |1               |529        |
|0610b486422d4921ae7d2bf64640c50b|55  |F      |112000.0         |0               |376        |
|38fe809add3b4fcf9315a9694bb96ff5|NULL|unknown|NULL             |1               |14         |
|78afa995795e4d85b5d9ceeca43f5fef|75  |F      |100000.0         |0               |443        |
|a03223e636434f42ac4c3df47e8bac43|NULL|unknown|NULL             |1               |356        |
+--------------------------------+----+-------+-----------------+----------------+-----------+
only showing top 5 rows


In [7]:
n_sentinel = profile.filter(F.col("identity_missing") == 1).count()
n_age_null = profile.filter(F.col("age").isNull()).count()
n_total = profile.count()

print(f"clientes totais: {n_total}")
print(f"identity_missing=1: {n_sentinel}")
print(f"age null: {n_age_null}")
print("sentinela e age-null coincidem?", n_sentinel == n_age_null)

profile.groupBy("gender").count().orderBy(F.desc("count")).show()

clientes totais: 17000
identity_missing=1: 2175
age null: 2175
sentinela e age-null coincidem? True


+-------+-----+
| gender|count|
+-------+-----+
|      M| 8484|
|      F| 6129|
|unknown| 2175|
|      O|  212|
+-------+-----+



In [8]:
profile.select(
    F.min("tenure_days").alias("tenure_min"),
    F.max("tenure_days").alias("tenure_max"),
    F.sum((F.col("tenure_days") < 0).cast("int")).alias("tenure_negativo"),
).show()

+----------+----------+---------------+
|tenure_min|tenure_max|tenure_negativo|
+----------+----------+---------------+
|         0|      1823|              0|
+----------+----------+---------------+



## 4. Atribuição temporal oferta→transação (T-104)

Grão = `offer received`. Cada linha ganha, no máximo, uma view pós-recebimento e
uma transação atribuída dentro de `[received_time, received_time + duration]`,
sob a regra de prioridade da config quando há sobreposição (Premissa 1).

In [9]:
attributed = attribution.attribute(events, offers, cfg)
attributed.show(5, truncate=False)

Premissa 1 violada em 17227 transação(ões): mais de uma oferta ativa no intervalo; prioridade 'earliest_received' aplicada.


+--------------------------------+--------------------------------+-------------+-----------+---------+------------------+-----------------------+-----------------------+
|account_id                      |offer_id                        |received_time|valid_until|view_time|assigned_txn_count|assigned_txn_amount_sum|first_assigned_txn_time|
+--------------------------------+--------------------------------+-------------+-----------+---------+------------------+-----------------------+-----------------------+
|e2127556f4f64592b11af22de27a7932|2906b810c7d4411798c6938adc9daaa5|0.0          |7.0        |0.75     |0                 |0.0                    |NULL                   |
|78afa995795e4d85b5d9ceeca43f5fef|9b98b8c7a33c4b65b9aebfe6a799e6d9|0.0          |7.0        |0.25     |2                 |37.67                  |5.5                    |
|68617ca6246f4fbc85e91a2a49552598|4d5c57ea9a6940dd891ad53e9dbe8da0|0.0          |5.0        |3.5      |0                 |0.0                    

In [10]:
n_received = events.filter(F.col("event") == "offer received").count()
n_attributed_rows = attributed.count()

print(f"eventos 'offer received': {n_received}")
print(f"linhas no grão attribution.attribute: {n_attributed_rows}")
print("grão preservado (== received)?", n_received == n_attributed_rows)

# checagem de duplicata na chave do contrato (prévia do G1)
dupes = (
    attributed.groupBy("account_id", "offer_id", "received_time")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print("duplicatas em (account_id, offer_id, received_time):", dupes)

eventos 'offer received': 76277
linhas no grão attribution.attribute: 76277
grão preservado (== received)? True


duplicatas em (account_id, offer_id, received_time): 0


In [11]:
attributed.select(
    F.sum(F.col("view_time").isNotNull().cast("int")).alias("com_view"),
    F.sum((F.col("assigned_txn_count") > 0).cast("int")).alias("com_txn_atribuida"),
    F.count("*").alias("total"),
).show()

+--------+-----------------+-----+
|com_view|com_txn_atribuida|total|
+--------+-----------------+-----+
|   56895|            39096|76277|
+--------+-----------------+-----+



## 5. Label influence-aware (T-105)

`converted=1` sse view precedente E transação atribuída dentro da validade —
nunca a partir de `offer completed` diretamente (cobre informational, G5).

In [12]:
labeled = attribution.build_label(attributed, cfg)
labeled.select(
    "account_id", "offer_id", "received_time", "valid_until",
    "view_time", "assigned_txn_count", "first_assigned_txn_time", "converted", "conversion_value",
).show(10, truncate=False)

+--------------------------------+--------------------------------+-------------+-----------+---------+------------------+-----------------------+---------+------------------+
|account_id                      |offer_id                        |received_time|valid_until|view_time|assigned_txn_count|first_assigned_txn_time|converted|conversion_value  |
+--------------------------------+--------------------------------+-------------+-----------+---------+------------------+-----------------------+---------+------------------+
|102e9454054946fda62242d2e176fdce|4d5c57ea9a6940dd891ad53e9dbe8da0|0.0          |5.0        |0.0      |4                 |0.25                   |1        |55.07             |
|25c906289d154b66bf579693f89481c9|2906b810c7d4411798c6938adc9daaa5|0.0          |7.0        |NULL     |0                 |NULL                   |0        |0.0               |
|fe8264108d5b4f198453bbb1fa7ca6c9|ae264e3637204a6fb9bb56bc8210ddfd|0.0          |7.0        |6.25     |1                

In [13]:
labeled.groupBy("converted").count().show()

conv_rate = labeled.select(F.avg("converted")).first()[0]
print(f"taxa de conversão influence-aware: {conv_rate:.4f}")

+---------+-----+
|converted|count|
+---------+-----+
|        1|39096|
|        0|37181|
+---------+-----+



taxa de conversão influence-aware: 0.5126


### Conversão por tipo de oferta

Cruza `labeled` com `offers` para checar se `informational` converte (via janela
pós-view) mesmo sem evento `offer completed` — não deveria haver diferença
estrutural no cálculo por tipo, já que `build_label` não olha para `offer completed`.

In [14]:
(
    labeled.join(offers.select(F.col("id").alias("offer_id"), "offer_type"), on="offer_id", how="left")
    .groupBy("offer_type")
    .agg(
        F.count("*").alias("n"),
        F.sum("converted").alias("conversoes"),
        F.avg("converted").alias("taxa_conversao"),
    )
    .orderBy("offer_type")
    .show(truncate=False)
)

+-------------+-----+----------+-------------------+
|offer_type   |n    |conversoes|taxa_conversao     |
+-------------+-----+----------+-------------------+
|bogo         |30499|17625     |0.5778877995999869 |
|discount     |30543|16843     |0.5514520512064958 |
|informational|15235|4628      |0.30377420413521494|
+-------------+-----+----------+-------------------+



### Completou sem ver (sure thing) — Premissa 2

Referência de contexto: a spec cita 28,4% dos pares (cliente, oferta) com
`offer completed` sem view precedente. Mede aqui a partir dos eventos brutos
para conferir a premissa nos dados carregados.

In [15]:
completed = events.filter(F.col("event") == "offer completed").select(
    F.col("account_id"), F.col("offer_ref").alias("offer_id"), F.col("time").alias("completed_time")
)
viewed_all = events.filter(F.col("event") == "offer viewed").select(
    F.col("account_id"), F.col("offer_ref").alias("offer_id"), F.col("time").alias("view_time")
)

completed_with_view = (
    completed.join(viewed_all, on=["account_id", "offer_id"], how="left")
    .withColumn("has_prior_view", (F.col("view_time").isNotNull()) & (F.col("view_time") <= F.col("completed_time")))
    .groupBy("account_id", "offer_id", "completed_time")
    .agg(F.max("has_prior_view").alias("has_prior_view"))
)

n_completed = completed_with_view.count()
n_sem_view = completed_with_view.filter(~F.col("has_prior_view")).count()
print(f"completed total: {n_completed}")
print(f"completed sem view precedente: {n_sem_view} ({n_sem_view / n_completed:.1%})")

completed total: 33182
completed sem view precedente: 8568 (25.8%)


## 6. Features anti-leakage (T-106)

Anexa ao grão rotulado as features `hist_*` (transacionais e de resposta a
ofertas), computadas só com eventos `time < received_time` (G2), mais as
features de oferta/contexto do catálogo. `features.build` recebe os eventos
parseados, o grão rotulado e o catálogo.

In [16]:
featured = features.build(events, labeled, offers, cfg)
print("colunas do grão com features:", len(featured.columns))
print("linhas (deve == 76277):", featured.count())

# grão ainda íntegro após todas as junções de feature?
dupes = (
    featured.groupBy("account_id", "offer_id", "received_time").count().filter(F.col("count") > 1).count()
)
print("duplicatas na chave do grão:", dupes)

colunas do grão com features: 38


linhas (deve == 76277): 76277


duplicatas na chave do grão: 0


In [17]:
feature_cols = [c for c in featured.columns if c.startswith("hist_")]
featured.select(
    "account_id", "offer_id",
    "hist_spend_total", "hist_txn_count", "hist_recency_days",
    "hist_offers_received", "hist_view_rate", "hist_completed_unseen_flag",
).show(8, truncate=False)

+--------------------------------+--------------------------------+----------------+--------------+-----------------+--------------------+--------------+--------------------------+
|account_id                      |offer_id                        |hist_spend_total|hist_txn_count|hist_recency_days|hist_offers_received|hist_view_rate|hist_completed_unseen_flag|
+--------------------------------+--------------------------------+----------------+--------------+-----------------+--------------------+--------------+--------------------------+
|2eeac8d8feae4a8cad5a6af0499a211d|3f207df678b143eea3cee63160fa8bed|0.0             |0             |NULL             |0                   |0.0           |0                         |
|e2127556f4f64592b11af22de27a7932|2906b810c7d4411798c6938adc9daaa5|0.0             |0             |NULL             |0                   |0.0           |0                         |
|78afa995795e4d85b5d9ceeca43f5fef|9b98b8c7a33c4b65b9aebfe6a799e6d9|0.0             |0          

### Checagem de leakage no dado real

Para uma amostra de linhas, confirma que nenhuma transação com
`time >= received_time` contribuiu para `hist_spend_total`. Recomputa a soma
de transações estritamente anteriores por cliente e compara.

In [18]:
txns = events.filter(F.col("event") == "transaction").select(
    "account_id", F.col("time").alias("txn_time"), F.col("amount").alias("txn_amount")
)
recheck = (
    featured.select("account_id", "offer_id", "received_time", "hist_spend_total")
    .join(txns, on="account_id", how="left")
    .filter(F.col("txn_time") < F.col("received_time"))
    .groupBy("account_id", "offer_id", "received_time", "hist_spend_total")
    .agg(F.sum("txn_amount").alias("recomputed_spend"))
    .withColumn("delta", F.abs(F.col("hist_spend_total") - F.col("recomputed_spend")))
)
max_delta = recheck.select(F.max("delta")).first()[0]
print(f"maior divergência entre hist_spend_total e recomputo anti-leakage: {max_delta}")
print("(esperado ~0 — dentro de erro de ponto flutuante)")

maior divergência entre hist_spend_total e recomputo anti-leakage: 0.0
(esperado ~0 — dentro de erro de ponto flutuante)


### Distribuição de features-chave

In [19]:
featured.select(
    "hist_spend_total", "hist_txn_count", "hist_avg_ticket",
    "hist_view_rate", "n_concurrent_offers",
).summary("min", "25%", "50%", "75%", "max", "mean").show()

+-------+------------------+------------------+------------------+------------------+-------------------+
|summary|  hist_spend_total|    hist_txn_count|   hist_avg_ticket|    hist_view_rate|n_concurrent_offers|
+-------+------------------+------------------+------------------+------------------+-------------------+
|    min|               0.0|                 0|               0.0|               0.0|                  0|
|    25%|               0.0|                 0|               0.0|               0.0|                  0|
|    50%|              16.2|                 3|              3.79|              0.75|                  1|
|    75%|             59.95|                 5|16.742727272727272|               1.0|                  1|
|    max|1325.7200000000003|                29|             962.1|               1.0|                  3|
|   mean| 43.81646092531172|3.4566383051247427| 9.763081933666522|0.5984691759420736| 0.7142782227932404|
+-------+------------------+------------------

## 7. Custo do desconto (T-107)

`reward_cost` = `discount_value` do catálogo quando a oferta converte e não é
informational; 0 caso contrário. Verifica G6: `reward_cost > 0` ⇒ `converted=1`
e `offer_type ≠ informational`.

In [20]:
priced = cost.add_reward_cost(featured, offers, cfg)

# G6: nenhuma violação
violations = priced.filter(
    (F.col("reward_cost") > 0)
    & ((F.col("converted") != 1) | (F.col("offer_type") == "informational"))
).count()
print("violações de G6 (deve ser 0):", violations)

priced.groupBy("offer_type", "converted").agg(
    F.count("*").alias("n"),
    F.round(F.avg("reward_cost"), 3).alias("reward_cost_medio"),
    F.round(F.sum("reward_cost"), 1).alias("reward_cost_total"),
).orderBy("offer_type", "converted").show(truncate=False)

violações de G6 (deve ser 0): 0


+-------------+---------+-----+-----------------+-----------------+
|offer_type   |converted|n    |reward_cost_medio|reward_cost_total|
+-------------+---------+-----+-----------------+-----------------+
|bogo         |0        |12874|0.0              |0.0              |
|bogo         |1        |17625|7.813            |137710.0         |
|discount     |0        |13700|0.0              |0.0              |
|discount     |1        |16843|2.7              |45483.0          |
|informational|0        |10607|0.0              |0.0              |
|informational|1        |4628 |0.0              |0.0              |
+-------------+---------+-----+-----------------+-----------------+



In [21]:
total_cost = priced.select(F.sum("reward_cost")).first()[0]
total_conv_value = priced.select(F.sum("conversion_value")).first()[0]
print(f"custo total de recompensa concedida: R$ {total_cost:,.2f}")
print(f"valor total de conversão atribuído:  R$ {total_conv_value:,.2f}")

custo total de recompensa concedida: R$ 183,193.00
valor total de conversão atribuído:  R$ 1,347,865.21


## Próximos passos

- T-108: contrato (`StructType` + Pydantic) e escrita em `data/processed/`,
  impondo o schema e validando amostra + garantias G1–G8.
- T-109/T-110/T-111: tema de figuras, EDA e check de balanço de covariáveis.

Este notebook deve ser descartado ou promovido a `notebooks/1_eda.ipynb` uma
vez que T-108 esteja pronto e o dataset processado possa ser lido diretamente
do disco em vez de recomputado aqui.